# FIRELINE — Validasi drill-down kabupaten/kota

Notebook ini menggabungkan temuan Kaka tanpa mengubah skor utama. Ia memeriksa bahwa payload checkpoint 3 memiliki hierarki provinsi → kabupaten/kota → grid-hari, serta menyiapkan ringkasan untuk dashboard.

In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'review_outputs'
grid = pd.read_csv(OUT / 'grid_day_priority.csv')
payload = json.loads((OUT / 'dashboard_payload.json').read_text(encoding='utf-8'))
print({'grid_day_rows': len(grid), 'boundaries': len(payload['boundaries']['features']), 'provinces': grid['province_name'].nunique(), 'kabupaten': grid['region_name'].nunique()})

{'grid_day_rows': 26773, 'boundaries': 56, 'provinces': 5, 'kabupaten': 57}


In [2]:
kabupaten_summary = (grid.groupby(['province_name', 'region_name'], dropna=False)
    .agg(grid_days=('grid_id', 'size'), unique_grids=('grid_id', 'nunique'),
         max_priority=('priority_score', 'max'), median_school_km=('nearest_school_km', 'median'))
    .reset_index().sort_values(['province_name', 'max_priority'], ascending=[True, False]))
kabupaten_summary.head(10)

,province_name,region_name,grid_days,unique_grids,max_priority,median_school_km
10,Kalimantan Barat,Mempawah,454,22,98.2812,2.390750
12,Kalimantan Barat,Sanggau,2147,121,95.9135,2.192117
14,Kalimantan Barat,Sintang,1633,155,92.1504,2.918199
6,Kalimantan Barat,Kota Singkawang,71,8,91.3515,1.497451
2,Kalimantan Barat,Kapuas Hulu,1063,139,90.6016,2.845118
11,Kalimantan Barat,Sambas,522,51,90.1531,2.929700
7,Kalimantan Barat,Kubu Raya,572,58,89.6053,1.839829
8,Kalimantan Barat,Landak,1180,83,87.7406,1.701060
4,Kalimantan Barat,Ketapang,2411,229,87.4262,3.087257
0,Kalimantan Barat,Bengkayang,650,61,84.9290,1.855021


In [3]:
properties = [f['properties'] for f in payload['boundaries']['features']]
assert all(('region_id' in p and 'region_name' in p) or ('kabupaten_code' in p and 'kabupaten_name' in p) for p in properties)
assert properties and len(kabupaten_summary) > 0
print('Boundary payload valid; score tetap berasal dari pipeline checkpoint 3.')

Boundary payload valid; score tetap berasal dari pipeline checkpoint 3.
